### 1 - Load Data

In [1]:
import numpy as np
import pandas as pd
import hnswlib
import time

# load dataset 
X = np.load("../0_data/processed/features.npy")
meta = pd.read_csv("../0_data/processed/meta.csv")

# query index 
query_idx = 10
query = X[query_idx].reshape(1, -1)

k = 20  # sama seperti faiss & annoy

print("Dataset shape:", X.shape)
print("Metadata shape:", meta.shape)

Dataset shape: (232725, 9)
Metadata shape: (232725, 3)


### 2 - Build HNSW

In [2]:
dim = X.shape[1]

index = hnswlib.Index(space='l2', dim=dim)

start = time.time()

index.init_index(
    max_elements=len(X),
    ef_construction=100,
    M=16
)

index.add_items(X)

build_time = time.time() - start

print(f"HNSW build time: {build_time:.4f} sec")

HNSW build time: 4.5403 sec


### 3 - Query Function

In [3]:
index.set_ef(50)

start = time.time()
indices, distances = index.knn_query(query, k=k)
query_time = time.time() - start

print(f"HNSW query time: {query_time:.6f} sec")

HNSW query time: 0.000000 sec


### 4 - Hasil Rekomendasi

In [4]:
query_genre = meta.iloc[query_idx]["genre"]

print("=== HNSW (Filtered) ===")
print("Query genre:", query_genre)

results = []

for i, dist in zip(indices[0], distances[0]):
    
    if i == query_idx:
        continue
    
    if meta.iloc[i]["genre"] != query_genre:
        continue
    
    results.append((i, dist))
    
    if len(results) >= 5:
        break

print("\n🎧 Recommendations (Same Genre):")

if len(results) == 0:
    print("(No same-genre results, showing closest matches)")

    for i in indices[0][:5]:
        if i == query_idx:
            continue
        print("-", meta.iloc[i]["track_name"], "-", meta.iloc[i]["genre"])

else:
    for i, dist in results:
        print("-", meta.iloc[i]["track_name"], "-", meta.iloc[i]["genre"], f"({dist:.4f})")

=== HNSW (Filtered) ===
Query genre: Movie

🎧 Recommendations (Same Genre):
- Boston - Movie (0.2011)
- The Tear Heals - Movie (0.2238)
- Do You Hear That? / I Wonder - Movie (0.2268)
- Pappy Shuffle (Maverick - Original Motion Picture Score) - Remastered - Movie (0.2527)
